# 부품 카테고리별 사용시간(UsageTime__c) 차이 검정

FRA/Field 사전 스캔: `part_category`별 `UsageTime__c` 중앙값이 크게 다름
(Cylinder 118h vs Washer 516h) → "초기불량(설계·조립 결함) vs 마모성(소재·내구)
결함"을 구분하는 첫 단서일 수 있어서 정식 검정.

**주 지표는 `UsageTime__c`**(고장 시점까지의 누적 사용시간, 결측 0%). 생산일~고장일
날짜 차이(`years_to_fail`)는 보조 지표로 맨 아래에 남겨둔다 — 사용시간이 "실제로
얼마나 굴렸는가"를 보는 반면, 날짜 차이는 "달력상 얼마나 버텼는가"라 방치·보관 같은
다른 요인이 섞일 수 있음.

## STEP 1. 표본 크기 재확인

`UsageTime__c`는 결측 0%라 `part_category`별 건수가 전체 데이터셋의 카테고리 분포와
그대로 같다. 어제 카이제곱 검정 때와 같은 이유로 **"No Part Info(부품정보 없음)"는
부품 카테고리가 아니라 결측 라벨이라 제외**하고, n>=20인 카테고리만 검정 대상으로
쓴다.

In [ ]:
import pandas as pd
import numpy as np

print('UsageTime__c 결측:', df['UsageTime__c'].isna().sum(), '/', len(df))
n_by_cat = df['part_category'].value_counts()


NO_PART = 'No Part Info(부품정보 없음)'
TARGET_CATS = n_by_cat[(n_by_cat.index != NO_PART) & (n_by_cat >= 20)].index.tolist()
print('검정 대상 카테고리 (No Part Info 제외, n>=20):')
for c in TARGET_CATS:
    print(f'  {c:<16} n={n_by_cat[c]}')
print('합계:', n_by_cat[TARGET_CATS].sum(), '건')
valid = df[df['part_category'].isin(TARGET_CATS)].copy()
len(valid)


검정 대상 카테고리 (No Part Info 제외, n>=20):
  Seal             n=427
  Bearing          n=353
  Gear             n=177
  Washer           n=116
  Cylinder         n=31
  Ring(Snap/C/O)   n=30
  Ball Joint       n=23
합계: 1157 건


1157

**Cylinder(31)와 Ball Joint(23)는 n>=20 기준은 넘지만 7개 중 가장 작은 축.**
아래 STEP들의 결과, 특히 이 둘이 낀 쌍별 비교는 "표본 상대적으로 작음" 캐비엇을
달고 해석한다.

---
## 부록: 보조 지표 — 생산일~고장일 날짜 차이 (`years_to_fail`)

본 검정(`UsageTime__c`)과 별개로, "사용시간은 짧은데 날짜는 오래 지나 고장"처럼
방치·보관 의심 사례가 있는지 가볍게 참고하는 용도. 이 섹션 자체가 본 검정 대상은
아니다.

### STEP 1. 파생 변수: `years_to_fail`

`fm_Glo_Kor_ProdDate__c`(생산일)와 `FailureDate__c`(고장일)를 날짜로 바꾸고 뺀다.

- `pd.to_datetime(..., errors='coerce')`: 날짜로 못 바꾸는 값은 에러 대신 `NaT`(결측)로
  처리.
- `.dt.days / 365`: Timedelta에서 일수만 뽑아 365로 나눠 "년" 단위로 변환.

In [3]:
df['prod_date'] = pd.to_datetime(df['fm_Glo_Kor_ProdDate__c'], errors='coerce')
df['fail_date'] = pd.to_datetime(df['FailureDate__c'], errors='coerce')
df['years_to_fail'] = (df['fail_date'] - df['prod_date']).dt.days / 365

total = len(df)
computed = df['years_to_fail'].notna().sum()
print(f'전체 {total}건 중 years_to_fail 계산 가능: {computed}건 ({computed/total*100:.1f}%)')
print(f'결측: {total-computed}건')
print('  prod_date 결측:', df['prod_date'].isna().sum())
print('  fail_date 결측:', df['fail_date'].isna().sum())

전체 1858건 중 years_to_fail 계산 가능: 1803건 (97.0%)
결측: 55건
  prod_date 결측: 54
  fail_date 결측: 1


### STEP 2. 이상치 확인: `years_to_fail < 0`이 있는가

In [4]:
neg = df[df['years_to_fail'] < 0]
print('음수 이상치:', len(neg), '건')
df['years_to_fail'].describe()

음수 이상치: 0 건


count    1803.000000
mean        3.135955
std         1.636338
min         0.263014
25%         1.991781
50%         2.835616
75%         3.997260
max        11.369863
Name: years_to_fail, dtype: float64

**확인 결과**: 0건. 최솟값 0.26년(약 96일), 최댓값 11.37년 — CLAUDE.md의
"관측기간 편중(생산 최대 11년 vs 클레임 기록 3년)"과 부합. 제외할 행 없음.